In [2]:
from a_slm import transformer
import torch
import torch.nn as nn
import importlib
from transformers import AutoTokenizer

importlib.reload(transformer)

/Users/desktop/Documents/a-slm/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'a_slm.transformer' from '/Users/desktop/Documents/a-slm/a_slm/transformer.py'>

In [3]:
torch.manual_seed(42)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M"
)

In [5]:
print("Vocab size:", len(tokenizer))

Vocab size: 49152


In [6]:
text = "The quick brown fox jumps over the lazy dog."

ids = tokenizer.encode(
    text,
    add_special_tokens = False
)

tokens = tokenizer.convert_ids_to_tokens(ids)

decoded = tokenizer.decode(
    ids,
    skip_special_tokens = True
)

print("Original:", text)
print("Tokens:  ", tokens)
print("IDs:     ", ids)
print("Decoded: ", decoded)

Original: The quick brown fox jumps over the lazy dog.
Tokens:   ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
IDs:      [504, 2365, 6354, 16438, 27003, 690, 260, 23790, 2767, 30]
Decoded:  The quick brown fox jumps over the lazy dog.


In [8]:
device = torch.device("mps")

documents = [
    "Neural networks learn useful representations from data.",
    "The Pacific Ocean is the largest ocean on Earth."
]

encoded_documents = []

for doc in documents:
    ids = tokenizer.encode(
        doc,
        add_special_tokens = False
    )

    # Explicitly mark the end of this document
    ids.append(tokenizer.eos_token_id)

    encoded_documents.append(ids)

token_stream = []

for ids in encoded_documents:
    token_stream.extend(ids)

T = 8

x_examples = []
y_examples = []

for i in range(0, len(token_stream) - T, T):
    chunk = token_stream[i:i + T + 1]

    if len(chunk) == T + 1:
        x_examples.append(chunk[:-1])
        y_examples.append(chunk[1:])

x = torch.tensor(
    x_examples,
    dtype=torch.long,
    device=device
)

y = torch.tensor(
    y_examples,
    dtype=torch.long,
    device=device
)

print("x shape:", x.shape)
print("y shape:", y.shape)

print("\nInput 0:")
print(tokenizer.convert_ids_to_tokens(x[0].tolist()))

print("\nTarget 0:")
print(tokenizer.convert_ids_to_tokens(y[0].tolist()))

x shape: torch.Size([2, 8])
y shape: torch.Size([2, 8])

Input 0:
['Ne', 'ural', 'Ġnetworks', 'Ġlearn', 'Ġuseful', 'Ġrepresentations', 'Ġfrom', 'Ġdata']

Target 0:
['ural', 'Ġnetworks', 'Ġlearn', 'Ġuseful', 'Ġrepresentations', 'Ġfrom', 'Ġdata', '.']


In [9]:
vocab_size = len(tokenizer)

model = transformer.Transformer(
    num_l4g_blocks = 6,
    hidden_size = 48,
    intermediate_size = 128,
    num_q_heads = 6,
    num_kv_heads = 2,
    vocab_size = vocab_size,
    window_size = 3
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 1e-3
)

loss_fn = nn.CrossEntropyLoss()

In [12]:
model.train()

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

for step in range(300):
    optimizer.zero_grad()

    logits = model(x)

    loss = loss_fn(
        logits.reshape(-1, vocab_size),
        y.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print(
            f"step {step:3d} | loss {loss.item():.4f}"
        )

Total parameters:     3,099,984
Trainable parameters: 3,099,984
step   0 | loss 27.6739
step  25 | loss 0.0019
step  50 | loss 0.0004
step  75 | loss 0.0003
step 100 | loss 0.0002
step 125 | loss 0.0002
step 150 | loss 0.0002
step 175 | loss 0.0002
step 200 | loss 0.0002
step 225 | loss 0.0001
step 250 | loss 0.0001
step 275 | loss 0.0001


In [14]:
model.eval()

prompt = "Cheese"

input_ids = tokenizer.encode(
    prompt,
    add_special_tokens=False,
    return_tensors="pt"
).to(device)

generated = input_ids

with torch.no_grad():
    for _ in range(20):
        logits = model(generated)

        # logits for the final position
        next_token_logits = logits[:, -1, :]

        # greedy choice: highest-scoring token
        next_token = torch.argmax(
            next_token_logits,
            dim=-1,
            keepdim=True
        )

        generated = torch.cat(
            [generated, next_token],
            dim=1
        )

        if next_token.item() == tokenizer.eos_token_id:
            break

print(
    tokenizer.decode(
        generated[0],
        skip_special_tokens=False
    )
)

Cheese<|endoftext|>
